# Tutorial 2: Training TinyGPT

This tutorial walks you through training a 2.5M-parameter transformer from
scratch using pure NumPy. By the end you'll have:

- A trained model that produces syntactically-plausible Python code
- A plot of the loss / match-rate curves
- An intuition for why 30 epochs is enough

**Difficulty:** Intermediate
**Estimated time:** 20 minutes (mostly training time)

## 1. Set up the trainer

We'll work with the project's TinyGPT model and trainer. The model is a
12-layer pre-LN transformer with BPE tokenizer — 2.5M parameters total.

In [ ]:
import sys
sys.path.insert(0, '../laboratory/python/lab_en')

import numpy as np
from tiny_gpt import TinyGPT, TinyGPTConfig
from tiny_gpt_trainer import BPETokenizer, TrainConfig

# Default config — 12 layers, hidden=128, vocab=512, BPE
config = TinyGPTConfig(
    vocab_size=512,
    hidden_dim=128,
    n_layers=12,
    n_heads=4,
    max_seq_len=32,
    mlp_ratio=4,
    use_mlp=True,
    use_layernorm=True,
    activation="gelu",
)
print(f"Total parameters: {TinyGPT(config).n_params:,}")
# Expected: Total parameters: 2,543,104

## 2. Build the BPE tokenizer

We'll train on the project's own source code (~2MB). BPE will learn 256
merge rules, giving a total vocab of 512 (256 bytes + 256 merges).

In [ ]:
# Load a small corpus (the project's own source code works well)
corpus_path = "../laboratory/python/lab_en/tiny_gpt.py"
with open(corpus_path) as f:
    corpus = f.read()

# Also include the trainer source
with open("../laboratory/python/lab_en/tiny_gpt_trainer.py") as f:
    corpus += "\n" + f.read()

print(f"Corpus size: {len(corpus):,} chars")

# Fit BPE with 256 merges (vocab=512 = 256 bytes + 256 merges)
tokenizer = BPETokenizer(vocab_size=512)
tokenizer.fit(corpus, n_merges=256)
print(f"BPE merges: {tokenizer.n_merges}")

# Verify roundtrip
sample = "def train(model):"
ids = tokenizer.encode(sample)
decoded = tokenizer.decode(ids)
assert decoded == sample, f"Roundtrip failed: {sample!r} -> {decoded!r}"
print(f"Roundtrip OK: {sample!r} -> {len(ids)} tokens -> {decoded!r}")

## 3. Configure training

We use Adam with cosine LR + 3-epoch warmup. The min_lr_ratio=0.1 keeps
the LR at 10% of peak (not zero) to avoid getting stuck.

In [ ]:
train_config = TrainConfig(
    epochs=30,
    batch_size=1,
    learning_rate=5e-4,
    weight_decay=1e-5,
    warmup_epochs=3,
    lr_schedule="cosine",
    min_lr_ratio=0.1,
    use_mlp=True,
    use_layernorm=True,
    seq_len=32,
    stride=64,
    eval_split=0.1,
    seed=42,
)
print(f"Training {config.n_layers} layers x {config.hidden_dim} hidden "
      f"for {train_config.epochs} epochs")
print(f"Optimizer: Adam (beta1=0.9, beta2=0.999, wd={train_config.weight_decay})")
print(f"LR schedule: cosine with {train_config.warmup_epochs}-epoch warmup")

## 4. Train!

This takes ~17 minutes on a CPU. For a quick demo, set `epochs=3` in the
config above. The trainer will print per-epoch progress.

In [ ]:
# NOTE: For a quick demo, uncomment the next line to set epochs=3
# train_config = TrainConfig(epochs=3, **{k: v for k, v in vars(train_config).items() if k != 'epochs'})

# from tiny_gpt_trainer import train
# history = train(
#     config=config,
#     train_config=train_config,
#     tokenizer=tokenizer,
#     corpus=corpus,
#     output_dir="results/notebook_run",
#     verbose=True,
# )
print("Training cell — uncomment to run (takes ~17 min for 30 epochs)")

## 5. Plot the training curves

After training, plot the loss / match-rate / LR curves.

In [ ]:
import matplotlib.pyplot as plt
import json
from pathlib import Path

# Load pre-trained history (from the repo's trained model)
hist_path = Path("../laboratory/python/lab_en/results/models/training_history.json")
if hist_path.exists():
    history = json.loads(hist_path.read_text())

    fig, (ax1, ax2, ax3) = plt.subplots(3, 1, figsize=(10, 9), sharex=True)

    epochs = [h["epoch"] for h in history["epochs"]]
    losses = [h["loss"] for h in history["epochs"]]
    mrs    = [h["match_rate"] * 100 for h in history["epochs"]]
    lrs    = [h["lr"] for h in history["epochs"]]

    ax1.semilogy(epochs, losses, 'b-o', ms=3)
    ax1.set_ylabel('Loss (log scale)')
    ax1.set_title('TinyGPT training - 30 epochs')
    ax1.grid(True, alpha=0.3)

    ax2.plot(epochs, mrs, 'g-o', ms=3)
    ax2.set_ylabel('Match rate (%)')
    ax2.set_ylim(0, max(mrs) * 1.2)
    ax2.grid(True, alpha=0.3)

    ax3.plot(epochs, lrs, 'r-o', ms=3)
    ax3.set_ylabel('Learning rate')
    ax3.set_xlabel('Epoch')
    ax3.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.savefig('training_curves.png', dpi=150)
    plt.show()
else:
    print("Run training first to generate training_history.json")

## 6. Inspect the weights

The trained weights are saved as `.npz` (10.2 MB for 2.5M float32 params).

In [ ]:
import numpy as np

weights_path = "../laboratory/python/lab_en/results/models/tiny_gpt_trained.npz"
weights = np.load(weights_path)
print("Weight keys (first 10):")
for k in list(weights.keys())[:10]:
    print(f"  {k}: {weights[k].shape}")

# Check weight statistics
for k in ["layers.0.attn_wq", "layers.0.W_fc1", "layers.0.ln1_gamma"]:
    if k in weights:
        w = weights[k]
        print(f"\n{k}: mean={w.mean():.4f}, std={w.std():.4f}, "
              f"min={w.min():.4f}, max={w.max():.4f}")

## 7. Generate text

Use the trained model to generate text from a prompt.

In [ ]:
from tiny_gpt_trainer import load_trained_model, generate_sample

model, tok = load_trained_model(
    weights_path,
    "../laboratory/python/lab_en/results/models/tiny_gpt_bpe.json",
)

prompts = ["def train(", "import numpy", "class Tiny", "# "]
for prompt in prompts:
    ids = tok.encode(prompt)
    out = generate_sample(model, ids, max_new_tokens=30, temperature=0.5)
    print(f"\nPrompt: {prompt!r}")
    print(f"Output: {tok.decode(out)!r}")

## Understanding the results

### Why does loss decay exponentially?

The corpus is small (~2MB), so the model is essentially **memorizing** it.
After 30 epochs, the loss is 0.0085 — well into the memorization regime.

### Why is match_rate only 6.5%?

Three reasons: small model (2.5M params), small corpus (2MB), small context
(32 tokens). 6.5% on a 512-token BPE vocab is 33× the random baseline.

### Why cosine LR + warmup?

- **Warmup** prevents early instability when gradients are large
- **Cosine decay** smoothly reduces LR to encourage convergence
- **min_lr_ratio=0.1** keeps LR at 10% of peak to avoid getting stuck

## What's next?

- [Tutorial 3: Generation & Sampling](03_generation_and_sampling.ipynb)
- [API: trainer](https://wild8highlander.github.io/rmt-llm-research/api/trainer/)